# Diet Data Enhancement Experiment Summary

This notebook summarizes downstream and Phenobench-adapter experiments for the Diet Data Enhancement project. It compares each diet representation against the basic NutriMatch baseline and highlights the strongest examples where enhanced diet representations improve prediction.

## What This Notebook Expects

Run this after the experiment code has produced one or more comparison CSVs. The most important expected file is:

`downstream_analysis/tasks/cvd/outputs/cvd_feature_set_comparison_memory_safe.csv`

If microbiome experiments are added later, the notebook will also pick up microbiome comparison CSVs when present.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
# Edit this only if you run the notebook from a different folder.
PROJECT_ROOT = Path.cwd()

tre_candidate = Path("/home/ec2-user/studies/Diet_Data_Enhancement_Project/Diet_Data_Enhancement_TRE")
if not (PROJECT_ROOT / "downstream_analysis").exists() and tre_candidate.exists():
    PROJECT_ROOT = tre_candidate

CVD_OUTPUT_DIR = PROJECT_ROOT / "downstream_analysis/tasks/cvd/outputs"
MICROBIOME_OUTPUT_DIR = PROJECT_ROOT / "downstream_analysis/tasks/microbiome_prediction/outputs"
ADAPTER_ROOT = PROJECT_ROOT / "outputs/phenobench_adapter"
REPORT_DIR = PROJECT_ROOT / "downstream_analysis/tasks/cvd/outputs/experiment_summary_notebook_outputs"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Report output:", REPORT_DIR)

## Load Experiment Tables

In [ ]:
comparison_candidates = [
    ("cvd", CVD_OUTPUT_DIR / "cvd_feature_set_comparison_memory_safe.csv"),
    ("cvd", CVD_OUTPUT_DIR / "cvd_feature_set_comparison.csv"),
    ("microbiome", MICROBIOME_OUTPUT_DIR / "microbiome_feature_set_comparison_memory_safe.csv"),
    ("microbiome", MICROBIOME_OUTPUT_DIR / "microbiome_feature_set_comparison.csv"),
]

frames = []
for analysis_type, path in comparison_candidates:
    if path.exists():
        frame = pd.read_csv(path)
        frame["analysis_type"] = analysis_type
        frame["source_file"] = str(path)
        frames.append(frame)
        print(f"Loaded {analysis_type}: {path} shape={frame.shape}")

if not frames:
    raise FileNotFoundError("No comparison CSVs found yet. Run the experiment first, then rerun this notebook.")

raw = pd.concat(frames, ignore_index=True)
raw.head()

In [ ]:
def coalesce_metric_columns(df):
    out = df.copy()
    aliases = {
        "r2_mean": ["r2_mean", "r2"],
        "r2_std": ["r2_std"],
        "rmse_mean": ["rmse_mean", "rmse"],
        "rmse_std": ["rmse_std"],
        "pearson_r_mean": ["pearson_r_mean", "pearson_r"],
        "pearson_r_std": ["pearson_r_std"],
    }
    for canonical, options in aliases.items():
        if canonical not in out.columns:
            for option in options:
                if option in out.columns:
                    out[canonical] = out[option]
                    break
    for col in ["r2_mean", "r2_std", "rmse_mean", "rmse_std", "pearson_r_mean", "pearson_r_std", "aligned_rows", "feature_count"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

df = coalesce_metric_columns(raw)

if "model" in df.columns and "ridge" in set(df["model"].dropna()):
    df = df[df["model"].eq("ridge")].copy()

df = df.dropna(subset=["feature_set", "target"])
print(df.shape)
df[[c for c in ["analysis_type", "feature_set", "target", "model", "r2_mean", "rmse_mean", "pearson_r_mean", "aligned_rows", "feature_count"] if c in df.columns]].head()

## Label Feature Sets

In [ ]:
FEATURE_GROUPS = {
    "basic_nutrimatch": "non-enhanced baseline",
    "nutrimatch_only": "non-enhanced baseline",
    "denovo_enriched": "enhanced food table",
    "enriched_data": "enhanced food table",
    "nutrimatch_enhanced": "enhanced food table",
    "full_data": "downstream enhanced",
    "denovo_cardiometabolic": "downstream enhanced",
    "denovo_broad_diet_health": "downstream enhanced",
    "denovo_microbiome": "downstream enhanced",
    "denovo_chemical_metabolomics": "downstream enhanced",
    "denovo_mental_health": "downstream enhanced",
    "nutrimatch_cardiometabolic": "downstream enhanced",
    "nutrimatch_broad_diet_health": "downstream enhanced",
    "nutrimatch_microbiome": "downstream enhanced",
    "nutrimatch_chemical_metabolomics": "downstream enhanced",
    "nutrimatch_mental_health": "downstream enhanced",
    "denovo_food_card_embedding": "embedding enhanced",
    "food_card_embedding": "embedding enhanced",
}

FEATURE_LABELS = {
    "basic_nutrimatch": "Basic NutriMatch",
    "nutrimatch_only": "Basic NutriMatch",
    "denovo_enriched": "De novo enriched",
    "enriched_data": "De novo enriched",
    "nutrimatch_enhanced": "NutriMatch enhanced",
    "full_data": "Full downstream",
    "denovo_cardiometabolic": "De novo cardiometabolic",
    "denovo_broad_diet_health": "De novo broad diet-health",
    "denovo_microbiome": "De novo microbiome-oriented",
    "denovo_chemical_metabolomics": "De novo chemical/metabolomics",
    "denovo_mental_health": "De novo mental-health",
    "nutrimatch_cardiometabolic": "NutriMatch cardiometabolic",
    "nutrimatch_broad_diet_health": "NutriMatch broad diet-health",
    "nutrimatch_microbiome": "NutriMatch microbiome-oriented",
    "nutrimatch_chemical_metabolomics": "NutriMatch chemical/metabolomics",
    "nutrimatch_mental_health": "NutriMatch mental-health",
    "denovo_food_card_embedding": "Food-card embedding",
    "food_card_embedding": "Food-card embedding",
}

TARGET_LABELS = {
    "bt__triglycerides_float_value": "Triglycerides",
    "bt__total_cholesterol_float_value": "Total cholesterol",
    "bt__hdl_cholesterol_float_value": "HDL cholesterol",
    "bt__ldl_cholesterol_float_value": "LDL cholesterol",
    "bt__non_hdl_cholesterol_float_value": "Non-HDL cholesterol",
    "bt__glucose_float_value": "Glucose",
    "bt__hba1c_float_value": "HbA1c",
    "bt__creatinine_float_value": "Creatinine",
    "bt__urate_float_value": "Urate",
    "bt__alt_float_value": "ALT",
    "bt__ast_float_value": "AST",
    "bt__ggt_float_value": "GGT",
}

df["feature_group"] = df["feature_set"].map(FEATURE_GROUPS).fillna("other")
df["feature_label"] = df["feature_set"].map(FEATURE_LABELS).fillna(df["feature_set"].astype(str))
df["target_label"] = df["target"].map(TARGET_LABELS).fillna(df["target"].astype(str))
df["is_enhanced"] = ~df["feature_group"].eq("non-enhanced baseline")

df[["feature_set", "feature_label", "feature_group"]].drop_duplicates().sort_values(["feature_group", "feature_set"])

## Overall Summary

In [ ]:
summary_cols = {
    "mean_r2": ("r2_mean", "mean"),
    "median_r2": ("r2_mean", "median"),
    "mean_rmse": ("rmse_mean", "mean"),
    "mean_pearson": ("pearson_r_mean", "mean"),
    "targets_tested": ("target", "nunique"),
    "rows_median": ("aligned_rows", "median"),
    "feature_count_median": ("feature_count", "median"),
}

available_aggs = {name: spec for name, spec in summary_cols.items() if spec[0] in df.columns}
overall = (
    df.groupby(["analysis_type", "feature_set", "feature_label", "feature_group"], dropna=False)
    .agg(**available_aggs)
    .reset_index()
    .sort_values(["analysis_type", "mean_r2"], ascending=[True, False])
)

overall_path = REPORT_DIR / "overall_feature_set_summary.csv"
overall.to_csv(overall_path, index=False)
print("Wrote:", overall_path)
overall

## Improvement Over Basic Diet Data

Positive `r2_delta_vs_baseline` means the feature set predicted the target better than the basic NutriMatch representation. Positive `rmse_improvement_vs_baseline` means lower error than baseline.

In [ ]:
def baseline_rows(frame):
    preferred = frame[frame["feature_set"].eq("basic_nutrimatch")]
    if not preferred.empty:
        return preferred
    fallback = frame[frame["feature_set"].eq("nutrimatch_only")]
    if not fallback.empty:
        return fallback
    raise ValueError("No baseline feature set found. Expected basic_nutrimatch or nutrimatch_only.")

key_cols = ["analysis_type", "target"]
if "model" in df.columns:
    key_cols.append("model")

base = baseline_rows(df)
base_keep = key_cols + ["r2_mean", "rmse_mean", "pearson_r_mean", "feature_set", "feature_label"]
base = base[[c for c in base_keep if c in base.columns]].rename(columns={
    "feature_set": "baseline_feature_set",
    "feature_label": "baseline_feature_label",
    "r2_mean": "baseline_r2_mean",
    "rmse_mean": "baseline_rmse_mean",
    "pearson_r_mean": "baseline_pearson_r_mean",
})

delta = df.merge(base, on=key_cols, how="left")
delta["r2_delta_vs_baseline"] = delta["r2_mean"] - delta["baseline_r2_mean"]
delta["rmse_improvement_vs_baseline"] = delta["baseline_rmse_mean"] - delta["rmse_mean"]
delta["pearson_delta_vs_baseline"] = delta["pearson_r_mean"] - delta["baseline_pearson_r_mean"]

delta = delta[~delta["feature_set"].eq(delta["baseline_feature_set"])]

delta_path = REPORT_DIR / "feature_set_deltas_vs_basic_nutrimatch.csv"
delta.to_csv(delta_path, index=False)
print("Wrote:", delta_path)

delta[[c for c in ["analysis_type", "target_label", "feature_label", "feature_group", "r2_mean", "baseline_r2_mean", "r2_delta_vs_baseline", "rmse_improvement_vs_baseline", "aligned_rows", "feature_count"] if c in delta.columns]].sort_values("r2_delta_vs_baseline", ascending=False).head(25)

In [ ]:
delta_summary = (
    delta.groupby(["analysis_type", "feature_set", "feature_label", "feature_group"], dropna=False)
    .agg(
        mean_r2_delta=("r2_delta_vs_baseline", "mean"),
        median_r2_delta=("r2_delta_vs_baseline", "median"),
        best_r2_delta=("r2_delta_vs_baseline", "max"),
        improved_targets=("r2_delta_vs_baseline", lambda s: int((s > 0).sum())),
        tested_targets=("r2_delta_vs_baseline", "count"),
        mean_rmse_improvement=("rmse_improvement_vs_baseline", "mean"),
    )
    .reset_index()
    .sort_values(["analysis_type", "mean_r2_delta"], ascending=[True, False])
)

delta_summary_path = REPORT_DIR / "delta_summary_vs_basic_nutrimatch.csv"
delta_summary.to_csv(delta_summary_path, index=False)
print("Wrote:", delta_summary_path)
delta_summary

## Best Examples For The Thesis

These are the target/feature-set pairs where enhanced data most clearly beats the basic diet representation.

In [ ]:
best_examples = (
    delta[delta["is_enhanced"] & delta["r2_delta_vs_baseline"].notna()]
    .sort_values("r2_delta_vs_baseline", ascending=False)
    .head(20)
    .copy()
)

best_examples["example"] = best_examples["target_label"] + " | " + best_examples["feature_label"]

best_path = REPORT_DIR / "best_examples_enhancement_works.csv"
best_examples.to_csv(best_path, index=False)
print("Wrote:", best_path)

best_examples[[c for c in ["analysis_type", "target_label", "feature_label", "feature_group", "r2_mean", "baseline_r2_mean", "r2_delta_vs_baseline", "rmse_mean", "baseline_rmse_mean", "rmse_improvement_vs_baseline", "aligned_rows", "feature_count"] if c in best_examples.columns]]

## Plots

In [ ]:
plot_df = overall.copy()
plot_df = plot_df.sort_values("mean_r2", ascending=True)

fig_h = max(5, 0.36 * len(plot_df))
fig, ax = plt.subplots(figsize=(11, fig_h))
sns.barplot(
    data=plot_df,
    x="mean_r2",
    y="feature_label",
    hue="feature_group",
    dodge=False,
    ax=ax,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Average prediction performance by diet representation")
ax.set_xlabel("Mean R2 across tested targets")
ax.set_ylabel("")
ax.legend(title="Feature group", loc="lower right")
plt.tight_layout()

path = REPORT_DIR / "average_r2_by_feature_set.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("Wrote:", path)

In [ ]:
heat = delta[delta["r2_delta_vs_baseline"].notna()].copy()

# Keep the clearest rows if many feature sets are present.
top_features = (
    heat.groupby("feature_label")["r2_delta_vs_baseline"]
    .mean()
    .sort_values(ascending=False)
    .head(14)
    .index
)
heat = heat[heat["feature_label"].isin(top_features)]

pivot = heat.pivot_table(
    index="feature_label",
    columns="target_label",
    values="r2_delta_vs_baseline",
    aggfunc="mean",
)
row_order = heat.groupby("feature_label")["r2_delta_vs_baseline"].mean().sort_values(ascending=False).index
pivot = pivot.reindex(row_order)

fig, ax = plt.subplots(figsize=(14, max(5, 0.42 * len(pivot))))
sns.heatmap(
    pivot,
    center=0,
    cmap="RdBu_r",
    linewidths=0.4,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Delta R2 vs basic NutriMatch"},
)
ax.set_title("Where enhanced diet representations improve prediction")
ax.set_xlabel("Target")
ax.set_ylabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()

path = REPORT_DIR / "delta_r2_heatmap_vs_basic_nutrimatch.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("Wrote:", path)

In [ ]:
top = best_examples.head(12).copy()
top = top.sort_values("r2_delta_vs_baseline", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(5, 0.48 * len(top))))
sns.barplot(
    data=top,
    x="r2_delta_vs_baseline",
    y="example",
    hue="feature_group",
    dodge=False,
    ax=ax,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Best examples that support the enhancement thesis")
ax.set_xlabel("Delta R2 vs basic NutriMatch")
ax.set_ylabel("")
ax.legend(title="Feature group", loc="lower right")
plt.tight_layout()

path = REPORT_DIR / "best_examples_delta_r2.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("Wrote:", path)

In [ ]:
winners = (
    df.sort_values("r2_mean", ascending=False)
    .drop_duplicates(["analysis_type", "target"])
    .copy()
)

winners = winners.merge(
    base[["analysis_type", "target", "baseline_r2_mean"] + (["model"] if "model" in base.columns else [])],
    on=["analysis_type", "target"] + (["model"] if "model" in base.columns else []),
    how="left",
)
winners["winner_delta_vs_baseline"] = winners["r2_mean"] - winners["baseline_r2_mean"]
winners_path = REPORT_DIR / "best_feature_set_per_target.csv"
winners.to_csv(winners_path, index=False)
print("Wrote:", winners_path)

fig, ax = plt.subplots(figsize=(8, 7))
sns.scatterplot(
    data=winners,
    x="baseline_r2_mean",
    y="r2_mean",
    hue="feature_group",
    style="analysis_type",
    s=90,
    ax=ax,
)
lims = [
    np.nanmin([winners["baseline_r2_mean"].min(), winners["r2_mean"].min(), 0]),
    np.nanmax([winners["baseline_r2_mean"].max(), winners["r2_mean"].max(), 0]),
]
pad = (lims[1] - lims[0]) * 0.08 if lims[1] > lims[0] else 0.1
lims = [lims[0] - pad, lims[1] + pad]
ax.plot(lims, lims, color="black", linestyle="--", linewidth=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_title("Best enhanced result per target vs baseline")
ax.set_xlabel("Basic NutriMatch R2")
ax.set_ylabel("Best observed R2")

for _, row in winners.iterrows():
    ax.annotate(row["target_label"], (row["baseline_r2_mean"], row["r2_mean"]), xytext=(4, 4), textcoords="offset points", fontsize=8)

plt.tight_layout()
path = REPORT_DIR / "winner_vs_baseline_scatter.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
print("Wrote:", path)

winners[[c for c in ["analysis_type", "target_label", "feature_label", "feature_group", "baseline_r2_mean", "r2_mean", "winner_delta_vs_baseline", "rmse_mean", "aligned_rows", "feature_count"] if c in winners.columns]].sort_values("winner_delta_vs_baseline", ascending=False)

## Optional: Inspect Phenobench Adapter Outputs

This checks whether adapter manifests exist for each feature set. It does not require Phenobench runs to have finished.

In [ ]:
adapter_rows = []
for manifest_path in sorted(ADAPTER_ROOT.glob("*/phenobench_adapter_manifest.json")):
    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)
    adapter_rows.append({
        "feature_set": manifest.get("feature_set_name"),
        "adapter_mode": manifest.get("adapter_mode"),
        "participant_count": manifest.get("participant_count"),
        "embedding_dim": manifest.get("embedding_dim"),
        "configs": len(manifest.get("phenobench_configs", [])),
        "embedding_artifact": manifest.get("embedding_artifact"),
        "manifest_path": str(manifest_path),
    })

adapter_summary = pd.DataFrame(adapter_rows)
if adapter_summary.empty:
    print("No adapter manifests found yet at", ADAPTER_ROOT)
else:
    adapter_path = REPORT_DIR / "phenobench_adapter_summary.csv"
    adapter_summary.to_csv(adapter_path, index=False)
    print("Wrote:", adapter_path)
    display(adapter_summary)

## Optional: Inspect Phenobench Run Manifests

If Phenobench has been run, this section scans `downstream_analysis/phenobench_adapter/phenobench_runs/experiments/*/*/manifest.json` and extracts the task, predictor, strategy, and primary metric when available.

In [ ]:
def flatten_phenobench_manifest(path):
    with path.open("r", encoding="utf-8") as f:
        m = json.load(f)

    metrics = m.get("metrics") or {}
    run = m.get("run") or {}
    task = m.get("task") or {}
    predictor = m.get("predictor") or {}
    strategy = m.get("strategy") or {}

    row = {
        "run_id": run.get("run_id"),
        "task": task.get("name"),
        "predictor": predictor.get("name"),
        "strategy": strategy.get("name"),
        "cohort_n": task.get("cohort_n"),
        "primary_metric_name": metrics.get("primary_metric_name"),
        "primary_metric_value": metrics.get("primary_metric_value"),
        "manifest_path": str(path),
    }

    for key, value in metrics.items():
        if isinstance(value, (int, float, str, bool)) or value is None:
            row[f"metric__{key}"] = value
    return row

phenobench_manifest_paths = sorted((PROJECT_ROOT / "experiments").glob("*/*/manifest.json"))
phenobench_rows = []
for path in phenobench_manifest_paths:
    try:
        phenobench_rows.append(flatten_phenobench_manifest(path))
    except Exception as exc:
        print("Could not parse", path, repr(exc))

phenobench_runs = pd.DataFrame(phenobench_rows)
if phenobench_runs.empty:
    print("No Phenobench run manifests found yet.")
else:
    runs_path = REPORT_DIR / "phenobench_run_manifest_summary.csv"
    phenobench_runs.to_csv(runs_path, index=False)
    print("Wrote:", runs_path)
    display(phenobench_runs.sort_values("primary_metric_value", ascending=False, na_position="last"))

## Short Written Takeaway Template

Use the tables above to fill in the exact best examples after the run is complete:

> Across the tested participant-level diet representations, enhanced diet features outperformed the basic NutriMatch baseline on the clearest targets shown in `best_examples_enhancement_works.csv`. The strongest examples were selected by improvement in R2 against the same target, model, and participant-alignment setup. This supports the thesis that the enhanced diet representations carry phenotype-relevant signal beyond the non-enhanced diet baseline.